In [ ]:
Company myCompany = new Company("Tiger-Soft Solutions");
Employee dev1 = new Employee("Антон", "Смирнов", "Senior C# Разработчик");
Employee manager1 = new Employee("Елена", "Козлова", "Менеджер проектов");
myCompany.AddEmployee(dev1);
myCompany.AddEmployee(manager1);
Customer cust1 = new Customer("Иван", "Иванов", "+7-999-111-22-33");
Customer cust2 = new Customer("Анна", "Петрова", "+7-999-444-55-66");
Order order1 = myCompany.CreateOrder("Разработка CRM-системы", cust1);
Order order2 = myCompany.CreateOrder("Настройка облачного сервера", cust2);
Order order3 = myCompany.CreateOrder("Мобильное приложение", cust1);
Console.WriteLine();
myCompany.AssignEmployeeToOrder(order1.OrderId, dev1);
myCompany.AssignEmployeeToOrder(order2.OrderId, manager1);
myCompany.AssignEmployeeToOrder(order3.OrderId, dev1);
Console.WriteLine("\n--- Изменение статусов и уведомления ---");
order1.ChangeStatus(OrderStatus.InProgress);
order1.ChangeStatus(OrderStatus.Completed);
order2.ChangeStatus(OrderStatus.InProgress);
order3.ChangeStatus(OrderStatus.InProgress);
order3.ChangeStatus(OrderStatus.Completed);
myCompany.ShowOrdersByStatus(OrderStatus.Completed);
myCompany.ShowOrdersByStatus(OrderStatus.InProgress);
myCompany.FindOrdersByCustomer(cust1);
myCompany.GenerateEmployeesReport();
public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}
public delegate void OrderStatusChangedHandler(Order order, OrderStatus oldStatus, OrderStatus newStatus);

public abstract class Person
{ 
    public string FirstName { get; set; }
    public string LastName { get; set; }

    protected Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }
    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }
    public virtual string GetInfo()
    {
        return $"Человек: {GetFullName()}";
    }
    public void UpdateName(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }
}
public class Employee : Person
{
    public string Position { get; set; }
    private int _ordersProcessed;
    public int OrdersProcessed
    {
        get { return _ordersProcessed; }
        set { _ordersProcessed = value < 0 ? 0 : value; }
    }
    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }
    public override string GetInfo()
    {
        return $"Сотрудник: {GetFullName()} | Должность: {Position} | Обработано заказов: {OrdersProcessed}";
    }
    public void IncrementProcessedOrders()
    {
        OrdersProcessed++;
    }
    public void Promote(string newPosition)
    {
        Position = newPosition;
        Console.WriteLine($"Сотрудник {GetFullName()} повышен до должности '{Position}'.");
    }
    public void ReceiveNotification(Order order, OrderStatus oldStatus, OrderStatus newStatus)
    {
        Console.WriteLine($"[Уведомление для {GetFullName()}]: Заказ #{order.OrderId} ('{order.Description}') сменил статус: {oldStatus} -> {newStatus}");
    }
}
public class Customer : Person
{
    public string PhoneNumber { get; set; }

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }
    public override string GetInfo()
    {
        return $"Заказчик: {GetFullName()} | Телефон: {PhoneNumber}";
    }
    public void UpdatePhone(string newPhone)
    {
        PhoneNumber = newPhone;
    }
    public bool HasValidPhone()
    {
        return !string.IsNullOrWhiteSpace(PhoneNumber);
    }
}
public class Order
{
    private static int _nextId = 1;
    public int OrderId { get; private set; }
    public string Description { get; set; }
    public Employee AssignedEmployee { get; set; }
    public Customer OrderCustomer { get; set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }
    public event OrderStatusChangedHandler OnStatusChanged;
    public Order(string description, Customer customer)
    {
        OrderId = _nextId++;
        Description = description;
        OrderCustomer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }
    public void AssignEmployee(Employee employee)
    {
        if (AssignedEmployee != null)
        {
            OnStatusChanged -= AssignedEmployee.ReceiveNotification;
        }
        AssignedEmployee = employee;
        OnStatusChanged += employee.ReceiveNotification;
    }
    public void ChangeStatus(OrderStatus newStatus)
    {
        if (Status == newStatus) return;

        OrderStatus oldStatus = Status;
        Status = newStatus;

        if (newStatus == OrderStatus.Completed && AssignedEmployee != null)
        {
            AssignedEmployee.IncrementProcessedOrders();
        }

        OnStatusChanged?.Invoke(this, oldStatus, newStatus);
    }
    public string GetOrderInfo()
    {
        string empName = AssignedEmployee != null ? AssignedEmployee.GetFullName() : "Не назначен";
        return $"Заказ #{OrderId} | '{Description}' | Статус: {Status} | Клиент: {OrderCustomer.GetFullName()} | Ответственный: {empName} | Дата: {CreatedAt:dd.MM.yyyy HH:mm}";
    }
    public void UpdateDescription(string newDescription)
    {
        Description = newDescription;
    }
}
public class Company
{
    public string Name { get; set; }
    public List<Employee> Employees { get; private set; } = new List<Employee>();
    public List<Order> Orders { get; private set; } = new List<Order>();
    public List<Customer> Customers { get; private set; } = new List<Customer>();
    public Company(string name)
    {
        Name = name;
    }
    public void AddEmployee(Employee employee)
    {
        Employees.Add(employee);
        Console.WriteLine($"В компанию '{Name}' принят сотрудник: {employee.GetFullName()} ({employee.Position})");
    }
    public void AddCustomer(Customer customer)
    {
        Customers.Add(customer);
    }
    public Order CreateOrder(string description, Customer customer)
    {
        if (!Customers.Contains(customer))
        {
            AddCustomer(customer);
        }
        Order newOrder = new Order(description, customer);
        Orders.Add(newOrder);
        Console.WriteLine($"Создан заказ #{newOrder.OrderId}: '{description}' для клиента {customer.GetFullName()}");
        return newOrder;
    }
    public void AssignEmployeeToOrder(int orderId, Employee employee)
    {
        Order order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order != null)
        {
            order.AssignEmployee(employee);
            Console.WriteLine($"На заказ #{orderId} назначен ответственный: {employee.GetFullName()}");
        }
    }
    public void ShowOrdersByStatus(OrderStatus status)
    {
        Console.WriteLine($"\n--- Заказы со статусом [{status}] ---");
        var filtered = Orders.Where(o => o.Status == status).ToList();
        if (filtered.Count == 0)
        {
            Console.WriteLine("Заказов с таким статусом нет.");
            return;
        }
        foreach (var order in filtered)
        {
            Console.WriteLine(order.GetOrderInfo());
        }
    }
    public void FindOrdersByCustomer(Customer customer)
    {
        Console.WriteLine($"\n--- Заказы клиента: {customer.GetFullName()} ---");
        var customerOrders = Orders.Where(o => o.OrderCustomer == customer).ToList();
        if (customerOrders.Count == 0)
        {
            Console.WriteLine("У данного клиента нет заказов.");
            return;
        }
        foreach (var order in customerOrders)
        {
            Console.WriteLine(order.GetOrderInfo());
        }
    }
    public void GenerateEmployeesReport()
    {
        Console.WriteLine($"\n--- Отчет по сотрудникам компании '{Name}' ---");
        foreach (var emp in Employees)
        {
            Console.WriteLine($"- {emp.GetFullName()} ({emp.Position}): обработано заказов {emp.OrdersProcessed}");
        }
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

В компанию 'Tiger-Soft Solutions' принят сотрудник: Антон Смирнов (Senior C# Разработчик)
В компанию 'Tiger-Soft Solutions' принят сотрудник: Елена Козлова (Менеджер проектов)

Создан заказ #1: 'Разработка CRM-системы' для клиента Иван Иванов
Создан заказ #2: 'Настройка облачного сервера' для клиента Анна Петрова
Создан заказ #3: 'Мобильное приложение' для клиента Иван Иванов

На заказ #1 назначен ответственный: Антон Смирнов
На заказ #2 назначен ответственный: Елена Козлова
На заказ #3 назначен ответственный: Антон Смирнов

--- Изменение статусов и уведомления ---
[Уведомление для Антон Смирнов]: Заказ #1 ('Разработка CRM-системы') сменил статус: Created -> InProgress
[Уведомление для Антон Смирнов]: Заказ #1 ('Разработка CRM-системы') сменил статус: InProgress -> Completed
[Уведомление для Елена Козлова]: Заказ #2 ('Настройка облачного сервера') сменил статус: Created -> InProgress
[Уведомление для Антон Смирнов]: Заказ #3 ('Мобильное приложение') сменил статус: Created -> InProgress